# 01 — Введение и краткий курс Python

**Книга:** «Data Science с нуля» — Джоэл Грасс (2-е издание, 2019)
**Главы:** 1 + 2
**Источник:** `scratch/introduction.py`, `scratch/crash_course_in_python.py`

Этот ноутбук — конспект-компиляция ключевого кода из глав 1-2 с подробными русскими комментариями. Каждая секция содержит: пояснение → определение → вызов с проверкой.

**Запуск:** `Kernel → Restart & Run All` (env `mlops`).


## Глава 1. Введение: поиск ключевых связей

Сквозной пример — маленькая социальная сеть из 10 пользователей. Цель главы: показать, что «дата сайенс» в основе — это просто структуры данных + итерации + `Counter` / `defaultdict`.


### 1.1. Моделирование данных

Пользователей храним как список словарей — это типичный приём, когда данные приходят «как есть» (JSON, API, таблица).

In [6]:
users = [
    {"id": 0, "name": "Hero"},
    {"id": 1, "name": "Dunn"},
    {"id": 2, "name": "Sue"},
    {"id": 3, "name": "Chi"},
    {"id": 4, "name": "Thor"},
    {"id": 5, "name": "Clive"},
    {"id": 6, "name": "Hicks"},
    {"id": 7, "name": "Devin"},
    {"id": 8, "name": "Kate"},
    {"id": 9, "name": "Klein"},
]

# Связи (граф) — список пар id. «Сырое» представление:
# искать друзей конкретного юзера — O(n).
friendship_pairs = [
    (0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4),
    (4, 5), (5, 6), (5, 7), (6, 8), (7, 8), (8, 9),
]

# Список смежности — dict: id -> [friend_ids].
# Строим ОДИН РАЗ, дальше доступ O(1) + сразу длина.
friendships = {user["id"]: [] for user in users}
for i, j in friendship_pairs:
    friendships[i].append(j)
    friendships[j].append(i)        # связь симметрична

print(f"Граф построен. У id=0 друзья: {friendships[0]}")
print(f"Всего связей: {sum(len(v) for v in friendships.values()) // 2}")


Граф построен. У id=0 друзья: [1, 2]
Всего связей: 12


### 1.2. Базовая метрика — количество друзей

**Ключевой приём DS:** любой «топ-N» = `list of tuples` + `sort(key=lambda ..., reverse=True)`.

In [7]:
def number_of_friends(user):
    """Сколько друзей у пользователя?"""
    return len(friendships[user["id"]])


# Суммарно и в среднем по сети
total_connections = sum(number_of_friends(u) for u in users)
avg_connections = total_connections / len(users)
print(f"Всего связей: {total_connections}")        # 24
print(f"Среднее число друзей: {avg_connections}")  # 2.4

# Сортировка по числу друзей — универсальный приём «топ-N»
num_friends_by_id = [(user["id"], number_of_friends(user)) for user in users]
num_friends_by_id.sort(key=lambda id_friends: id_friends[1], reverse=True)

print("Топ по числу друзей (id, num_friends):")
for uid, n in num_friends_by_id:
    print(f"  user {uid:2d}: {n} друзей")


Всего связей: 24
Среднее число друзей: 2.4
Топ по числу друзей (id, num_friends):
  user  1: 3 друзей
  user  2: 3 друзей
  user  3: 3 друзей
  user  5: 3 друзей
  user  8: 3 друзей
  user  0: 2 друзей
  user  4: 2 друзей
  user  6: 2 друзей
  user  7: 2 друзей
  user  9: 1 друзей


### 1.3. Friends of friends (FOAF)

Идея: рекомендовать людям тех, кто дружит с их друзьями, но **не является прямым другом**. `Counter` сразу даёт «рейтинг кандидатов».

In [8]:
from collections import Counter


def friends_of_friends(user):
    """Счётчик FOAF для пользователя (исключая его и прямых друзей)."""
    user_id = user["id"]
    return Counter(
        foaf_id
        for friend_id in friendships[user_id]              # мои друзья
        for foaf_id in friendships[friend_id]              # их друзья
        if foaf_id != user_id                              # не я сам
        and foaf_id not in friendships[user_id]            # не прямой друг
    )


# Реальный вызов — то, чего не было в предыдущей версии!
print("FOAF для Hero (id=0):")
print(friends_of_friends(users[0]))
print("\nFOAF для Dunn (id=1):")
print(friends_of_friends(users[1]))
print("\nFOAF для Sue (id=2):")
print(friends_of_friends(users[2]))


FOAF для Hero (id=0):
Counter({3: 2})

FOAF для Dunn (id=1):
Counter({4: 1})

FOAF для Sue (id=2):
Counter({4: 1})


### 1.4. Группировка интересов через `defaultdict`

Два прохода: «интерес → пользователи» и «пользователь → интересы». Это **универсальный шаблон «инвертированного индекса»** в DS — так устроен поиск Google, рекомендательные системы, etc.

In [9]:
from collections import defaultdict

interests = [
    (0, "Hadoop"), (0, "Big Data"), (0, "HBase"), (0, "Java"),
    (0, "Spark"), (0, "Storm"), (0, "Cassandra"),
    (1, "NoSQL"), (1, "MongoDB"), (1, "Cassandra"), (1, "HBase"),
    (1, "Postgres"), (2, "Python"), (2, "scikit-learn"), (2, "scipy"),
    (2, "numpy"), (2, "statsmodels"), (2, "pandas"),
    (3, "R"), (3, "Python"), (3, "statistics"), (3, "regression"),
    (3, "probability"), (4, "machine learning"), (4, "regression"),
    (4, "decision trees"), (4, "libsvm"), (5, "Python"),
    (5, "Java"), (5, "R"), (5, "JavaScript"), (6, "statistics"),
    (6, "probability"), (6, "mathematics"), (6, "theory"),
    (7, "machine learning"), (7, "scikit-learn"), (7, "Mahout"),
    (7, "neural networks"), (8, "neural networks"), (8, "deep learning"),
    (8, "Big Data"), (8, "artificial intelligence"),
    (9, "Hadoop"), (9, "Java"), (9, "MapReduce"), (9, "Big Data"),
]

# Прямой индекс: интерес -> [user_id]
user_ids_by_interest = defaultdict(list)
for user_id, interest in interests:
    user_ids_by_interest[interest].append(user_id)

# Обратный индекс: user_id -> [interest]
interests_by_user_id = defaultdict(list)
for user_id, interest in interests:
    interests_by_user_id[user_id].append(interest)


def most_common_interests_with(user):
    """С кем у пользователя больше всего общих интересов?"""
    return Counter(
        interested_user_id
        for interest in interests_by_user_id[user["id"]]
        for interested_user_id in user_ids_by_interest[interest]
        if interested_user_id != user["id"]
    )


# Проверки
print("Прямой индекс (3 случайных интереса):")
for k in list(user_ids_by_interest.keys())[:3]:
    print(f"  {k!r} -> {user_ids_by_interest[k]}")

print("\nFOAF-интересы для Hero (id=0):")
print(most_common_interests_with(users[0]).most_common(3))


Прямой индекс (3 случайных интереса):
  'Hadoop' -> [0, 9]
  'Big Data' -> [0, 8, 9]
  'HBase' -> [0, 1]

FOAF-интересы для Hero (id=0):
[(9, 3), (1, 2), (8, 1)]


### 1.5. Бакетизация (binning) — зарплата по стажу

«Грязные» данные с пропусками — типичная задача. Стратегия: разбить непрерывную величину tenure на 3 бакета и посчитать среднюю зарплату в каждом.

In [10]:
salaries_and_tenures = [
    (83000, 8.7), (88000, 8.1), (48000, 0.7), (76000, 6),
    (69000, 6.5), (76000, 7.5), (60000, 2.5), (83000, 10),
    (48000, 1.9), (63000, 4.2), (None, 4.0),   # одна зарплата неизвестна
    (72000, 7.0), (71000, 7.9), (91000, 8.0), (79000, 7.0),
]


def tenure_bucket(tenure):
    """Сгруппировать стаж в 3 бакета."""
    if tenure < 2:
        return "less than two"
    elif tenure < 5:
        return "between two and five"
    else:
        return "more than five"


# Группировка: bucket -> [salary, ...]
salary_by_tenure_bucket = defaultdict(list)
for salary, tenure in salaries_and_tenures:
    if salary is not None:                          # пропуски отбрасываем
        salary_by_tenure_bucket[tenure_bucket(tenure)].append(salary)

# Средняя зарплата по бакетам
average_salary_by_bucket = {
    bucket: sum(salaries) / len(salaries)
    for bucket, salaries in salary_by_tenure_bucket.items()
}

print("Средняя зарплата по бакетам tenure:")
for bucket, avg in average_salary_by_bucket.items():
    n = len(salary_by_tenure_bucket[bucket])
    print(f"  {bucket!r}: ${avg:,.0f} (n={n})")


Средняя зарплата по бакетам tenure:
  'more than five': $78,800 (n=10)
  'less than two': $48,000 (n=2)
  'between two and five': $61,500 (n=2)


### 1.6. Подсчёт слов — `Counter.most_common`

Тот же приём, что в `word_count` библиотеках и в `tf-idf`.

In [11]:
words_and_counts = Counter(
    word
    for user_id, interest in interests
    for word in interest.lower().split()
)

print("Топ-10 слов в интересах:")
for word, count in words_and_counts.most_common(10):
    print(f"  {word!r}: {count}")


Топ-10 слов в интересах:
  'big': 3
  'data': 3
  'java': 3
  'python': 3
  'learning': 3
  'hadoop': 2
  'hbase': 2
  'cassandra': 2
  'scikit-learn': 2
  'r': 2


## Глава 2. Краткий курс Python

Эта глава — справочник по минимальному набору Python, нужному для DS. Ниже — самые важные приёмы с краткими примерами и **реальными** `assert`-проверками.


### 2.1. Функции

In [12]:
def double(x):
    """Докстринг — описание функции (показывается в help())."""
    return x * 2


def my_print(message="default"):
    print(message)


def full_name(first="Name", last="Surname"):
    return f"{first} {last}"


def apply_to_one(f):
    """Принимает функцию — основа функционального стиля."""
    return f(1)


# Демонстрация + assert
print(double(5))                        # 10
print(full_name(last="Grus"))            # "Name Grus"
print(apply_to_one(lambda x: x + 4))     # 5

assert double(7) == 14
assert full_name("Joel", "Grus") == "Joel Grus"
assert apply_to_one(lambda x: x * 10) == 10
print("[OK] 2.1 Функции")


10
Name Grus
5
[OK] 2.1 Функции


### 2.2. Строки

In [13]:
# f-строки (Python 3.6+) — главный способ форматирования
first_name, last_name = "Joel", "Grus"
full = f"{first_name} {last_name}"
print(full)

# Сырые строки — без экранирования
path = r"\t"   # два символа: '\' и 't' (НЕ табуляция)
print(f"path = {path!r}, len = {len(path)}")

# Старый способ форматирования (ещё встречается)
print("{0} {1}".format(first_name, last_name))

assert f"{first_name} {last_name}" == "Joel Grus"
assert path == "\\t"
print("[OK] 2.2 Строки")


Joel Grus
path = '\\t', len = 2
Joel Grus
[OK] 2.2 Строки


### 2.3. Исключения

In [14]:
try:
    result = 0 / 0
    print(result)
except ZeroDivisionError:
    print("cannot divide by zero")

print("После try/except — код не упал")


cannot divide by zero
После try/except — код не упал


### 2.4. Списки — самая частая структура

In [15]:
x = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# Индексация
print(f"x[0]={x[0]}, x[-1]={x[-1]}, x[-2]={x[-2]}")

# Срезы [start:end:step] — end не включается
print(f"x[:3]    = {x[:3]}")
print(f"x[3:]    = {x[3:]}")
print(f"x[1:5]   = {x[1:5]}")
print(f"x[::3]   = {x[::3]}")
print(f"x[5:2:-1]= {x[5:2:-1]}")

# Asserts
assert x[0] == 0 and x[-1] == 9 and x[-2] == 8
assert x[:3] == [0, 1, 2]
assert x[3:] == [3, 4, 5, 6, 7, 8, 9]
assert x[1:5] == [1, 2, 3, 4]
assert x[::3] == [0, 3, 6, 9]
assert x[5:2:-1] == [5, 4, 3]
print("[OK] 2.4 Списки")


x[0]=0, x[-1]=9, x[-2]=8
x[:3]    = [0, 1, 2]
x[3:]    = [3, 4, 5, 6, 7, 8, 9]
x[1:5]   = [1, 2, 3, 4]
x[::3]   = [0, 3, 6, 9]
x[5:2:-1]= [5, 4, 3]
[OK] 2.4 Списки


### 2.5. Деструктуризация

In [16]:
a, b = [1, 2]
_, y = [1, 2]                    # _ — игнорируем
a, b = b, a                       # Pythonic swap


def add(x, y):
    return x + y


print(f"a, b = {a}, {b}")
print(f"add(*[3, 4]) = {add(*[3, 4])}")  # * распаковывает список в аргументы

assert a == 2 and b == 1
assert add(*[3, 4]) == 7
print("[OK] 2.5 Деструктуризация")


a, b = 2, 1
add(*[3, 4]) = 7
[OK] 2.5 Деструктуризация


### 2.6. Кортежи — неизменяемые списки

In [17]:
my_tuple = (1, 2)
x, y = (3, 4)
print(f"my_tuple = {my_tuple}, x, y = {x}, {y}")

# my_tuple[0] = 99  → TypeError: кортежи нельзя менять

assert my_tuple == (1, 2)
assert (x, y) == (3, 4)
print("[OK] 2.6 Кортежи")


my_tuple = (1, 2), x, y = 3, 4
[OK] 2.6 Кортежи


### 2.7. Словари — ключевая DS-структура

In [18]:
grades = {"Joel": 80, "Tim": 95}

print(f"grades['Joel']      = {grades['Joel']}")             # 80
print(f"grades.get('Kate',0) = {grades.get('Kate', 0)}")     # 0 — безопасно
print(f"'Joel' in grades     = {'Joel' in grades}")          # проверка КЛЮЧА

tweet = {
    "user": "joelgrus", "text": "Data Science",
    "retweet_count": 100, "hashtags": ["#ds", "#python"],
}

print("\nTweet keys:    ", list(tweet.keys()))
print("Tweet values:  ", list(tweet.values()))

assert grades["Joel"] == 80
assert grades.get("Kate", 0) == 0
assert "Joel" in grades
print("[OK] 2.7 Словари")


grades['Joel']      = 80
grades.get('Kate',0) = 0
'Joel' in grades     = True

Tweet keys:     ['user', 'text', 'retweet_count', 'hashtags']
Tweet values:   ['joelgrus', 'Data Science', 100, ['#ds', '#python']]
[OK] 2.7 Словари


### 2.8. `defaultdict` и `Counter` — must-have коллекции

In [19]:
# defaultdict — dict с фабрикой значения по умолчанию
word_counts = defaultdict(int)        # int() -> 0
dd_list = defaultdict(list)           # list() -> []
dd_dict = defaultdict(dict)           # dict() -> {}

# Counter — подсчёт элементов
c = Counter([0, 1, 2, 0])             # Counter({0: 2, 1: 1, 2: 1})
print(f"Counter: {c}")
print(f"most_common(3): {c.most_common(3)}")

# Практический пример: подсчёт слов в списке интересов
all_interests = [word for _, interest in interests for word in interest.lower().split()]
interests_counter = Counter(all_interests)
print(f"\nСамые частые слова в интересах: {interests_counter.most_common(5)}")

assert c[0] == 2 and c.most_common(3)[0] == (0, 2)
print("[OK] 2.8 defaultdict/Counter")


Counter: Counter({0: 2, 1: 1, 2: 1})
most_common(3): [(0, 2), (1, 1), (2, 1)]

Самые частые слова в интересах: [('big', 3), ('data', 3), ('java', 3), ('python', 3), ('learning', 3)]
[OK] 2.8 defaultdict/Counter


### 2.9. Множества (set) — O(1) проверка членства

In [20]:
s = set()
s.add(1)
s.add(2)
s.add(2)                               # дубликат игнорируется
print(f"s = {s}, 2 in s = {2 in s}")

n_unique = len({1, 2, 2, 3})           # 3 — set оставляет уникальные
print(f"len({{1,2,2,3}}) = {n_unique}")

assert 2 in s
assert n_unique == 3
print("[OK] 2.9 Множества")


s = {1, 2}, 2 in s = True
len({1,2,2,3}) = 3
[OK] 2.9 Множества


### 2.10. List comprehension — САМЫЙ частый приём в DS-коде

Форма: `[expression for item in iterable if condition]`

In [21]:
even_numbers = [x for x in range(5) if x % 2 == 0]   # [0, 2, 4]
squares      = [x * x for x in range(5)]                  # [0, 1, 4, 9, 16]
pairs        = [(x, y) for x in range(3) for y in range(3)]
increasing   = [(x, y) for x in range(5) for y in range(x + 1, 5)]
square_dict  = {x: x * x for x in range(5)}              # dict comp
square_set   = {x * x for x in [1, -1]}                  # set comp
zeros        = [0 for _ in even_numbers]

print(f"even_numbers:  {even_numbers}")
print(f"squares:       {squares}")
print(f"pairs (3x3):   {pairs}")
print(f"increasing:    {increasing}")
print(f"square_dict:   {square_dict}")
print(f"square_set:    {square_set}")

assert even_numbers == [0, 2, 4]
assert squares == [0, 1, 4, 9, 16]
assert len(pairs) == 9
assert square_dict[3] == 9
assert square_set == {1}
print("[OK] 2.10 Comprehensions")


even_numbers:  [0, 2, 4]
squares:       [0, 1, 4, 9, 16]
pairs (3x3):   [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
increasing:    [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
square_dict:   {0: 0, 1: 1, 2: 4, 3: 9, 4: 16}
square_set:    {1}
[OK] 2.10 Comprehensions


### 2.11. Генераторы — ленивые последовательности (экономят память)

Generator expression — как list comp, но в круглых скобках. **НЕ вычисляется сразу!** Дайте ему `sum()` / `list()` / `for` — тогда посчитается.

In [22]:
def generate_range(n):
    i = 0
    while i < n:
        yield i
        i += 1


# Генератор материализуем в список ОДИН раз (после sum/list он исчерпывается!)
evens_below_20 = list(i for i in range(20) if i % 2 == 0)
print(f"evens_below_20      = {evens_below_20}")
print(f"sum(evens_below_20) = {sum(evens_below_20)}")
print(f"first 5 (через срез) = {evens_below_20[:5]}")

# Композиция генераторов — данные текут по цепочке без материализации


def natural_numbers():
    n = 1
    while True:
        yield n
        n += 1


data = natural_numbers()
evens = (x for x in data if x % 2 == 0)
even_squares = (x ** 2 for x in evens)
# Возьмём первые 5 значений
first_five = []
for i, val in enumerate(even_squares):
    if i >= 5:
        break
    first_five.append(val)
print(f"Первые 5 квадратов чётных: {first_five}")

assert sum(evens_below_20) == 90
assert first_five == [4, 16, 36, 64, 100]
print("[OK] 2.11 Генераторы")


evens_below_20      = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
sum(evens_below_20) = 90
first 5 (через срез) = [0, 2, 4, 6, 8]
Первые 5 квадратов чётных: [4, 16, 36, 64, 100]
[OK] 2.11 Генераторы


### 2.12. `enumerate` — обход с индексом

In [23]:
names = ["Alice", "Bob", "Charlie"]
for i, name in enumerate(names):
    print(f"  name {i} is {name}")

assert list(enumerate(names)) == [(0, "Alice"), (1, "Bob"), (2, "Charlie")]
print("[OK] 2.12 enumerate")


  name 0 is Alice
  name 1 is Bob
  name 2 is Charlie
[OK] 2.12 enumerate


### 2.13. `zip` — параллельная итерация

In [24]:
list1 = ['a', 'b', 'c']
list2 = [1, 2, 3]
zipped = list(zip(list1, list2))
print(f"zip: {zipped}")

# Обратная операция — распаковка zip
pairs = [('a', 1), ('b', 2), ('c', 3)]
letters, numbers = zip(*pairs)
print(f"letters: {letters}, numbers: {numbers}")

assert zipped == [('a', 1), ('b', 2), ('c', 3)]
assert letters == ('a', 'b', 'c') and numbers == (1, 2, 3)
print("[OK] 2.13 zip")


zip: [('a', 1), ('b', 2), ('c', 3)]
letters: ('a', 'b', 'c'), numbers: (1, 2, 3)
[OK] 2.13 zip


### 2.14. `*args` и `**kwargs` — гибкие сигнатуры

In [27]:
def magic(*args, **kwargs):
    print(f"  unnamed args (tuple): {args}")
    print(f"  keyword args (dict): {kwargs}")


magic(3, 4, key="word3", key2="word4")

# args — кортеж позиционных, kwargs — dict именованных


  unnamed args (tuple): (3, 4)
  keyword args (dict): {'key': 'word3', 'key2': 'word4'}


### 2.15. Декораторы — функции высшего порядка

In [26]:
def doubler(f):
    def g(*args, **kwargs):
        return 2 * f(*args, **kwargs)
    return g


@doubler
def add(x, y):
    return x + y


print(f"add(1, 2)        = {add(1, 2)}")      # 6
print(f"add(10, 100)     = {add(10, 100)}")   # 220

assert add(1, 2) == 6
print("[OK] 2.15 Декораторы")


add(1, 2)        = 6
add(10, 100)     = 220
[OK] 2.15 Декораторы


### 2.16. Сортировка с `key`

In [22]:
print(f"sorted([4,1,2,3]) = {sorted([4, 1, 2, 3])}")
print(f"sorted by abs desc = {sorted([-4, 1, -2, 3], key=abs, reverse=True)}")

x = [3, 1, 2]
x.sort()                                  # in-place
print(f"x after sort: {x}")

assert sorted([4, 1, 2, 3]) == [1, 2, 3, 4]
assert sorted([-4, 1, -2, 3], key=abs, reverse=True) == [-4, 3, -2, 1]
print("[OK] 2.16 Сортировка")


sorted([4,1,2,3]) = [1, 2, 3, 4]
sorted by abs desc = [-4, 3, -2, 1]
x after sort: [1, 2, 3]
[OK] 2.16 Сортировка


### 2.17. `random` — воспроизводимость через `seed`

In [31]:
import random

random.seed(10)
print(f"random()        = {random.random():.4f}")
print(f"randrange(10)   = {random.randrange(10)}")

lst = list(range(7))
random.shuffle(lst)
print(f"after shuffle   = {lst}")
print(f"choice          = {random.choice(lst)}")
print(f"sample(2)       = {random.sample(lst, 2)}")


random()        = 0.5714
randrange(10)   = 6
after shuffle   = [5, 2, 6, 1, 0, 4, 3]
choice          = 3
sample(2)       = [6, 4]


### 2.18. `re` — регулярные выражения

In [24]:
import re

print(f"match('a','cat')   = {re.match('a', 'cat')}")           # None
print(f"search('a','cat')  = {re.search('a', 'cat')}")          # match
print(f"split('[ab]','carbs') = {re.split('[ab]', 'carbs')}")
print(f"sub('[0-9]','-','R2D2') = {re.sub('[0-9]', '-', 'R2D2')}")

assert re.match("a", "cat") is None
assert re.search("a", "cat") is not None
assert re.split("[ab]", "carbs") == ['c', 'r', 's']
assert re.sub("[0-9]", "-", "R2D2") == "R-D-"
print("[OK] 2.18 re")


match('a','cat')   = None
search('a','cat')  = <re.Match object; span=(1, 2), match='a'>
split('[ab]','carbs') = ['c', 'r', 's']
sub('[0-9]','-','R2D2') = R-D-
[OK] 2.18 re


### 2.19. Type hints — самодокументирующийся код

In [32]:
from typing import List, Dict, Tuple, Set, Iterable, Callable, Optional, Union

Vector = List[float]


def total(xs: List[float]) -> float:
    return sum(xs)


def twice(repeater: Callable[[str, int], str], s: str) -> str:
    """Сигнатура: f принимает (str, int) -> str."""
    return repeater(s, 2)


print(f"total([1,2,3,4]) = {total([1.0, 2.0, 3.0, 4.0])}")
print(f"twice(lambda s, n: s*n, 'ab') = {twice(lambda s, n: s * n, 'ab')}")

values: List[int] = []
best_so_far: Optional[float] = None
operation: Union[str, int, float, bool] = "init"

assert total([1.0, 2.0, 3.0, 4.0]) == 10.0
assert twice(lambda s, n: s * n, "ab") == "abab"
print("[OK] 2.19 Type hints")


total([1,2,3,4]) = 10.0
twice(lambda s, n: s*n, 'ab') = abab
[OK] 2.19 Type hints


## Итог глав 1-2

Главный вывод: вся дальнейшая книга строится на:
1. `defaultdict` + `Counter` для группировок и подсчётов
2. List comprehensions / generators для преобразований
3. `zip` / `enumerate` для итераций
4. lambda / decorators / `*args` / type hints — для чистого кода

Эти 4 кита покрывают ~90% всего кода в DS-практике.
